# 01 · EDA — Fleet telemetry

In [17]:
import json
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

# Adjust base_dir if your notebook is inside a subfolder like 'notebooks/'
DATA_PATH = Path("../data/device-telemetry-dataset.ndjson") 

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Still missing at: {DATA_PATH.resolve()}")

with DATA_PATH.open("r", encoding="utf-8") as f:
    raw = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(raw)} records.")

Loaded 750 records.


In [20]:
def all_paths(obj, prefix=""):
    """Every key at every depth, parents included.

    dict key     -> 'battery', 'battery.condition', ...
    list field   -> 'network' (key exists), 'network[]' (list has >= 1 item),
                    'network[].mac' (key appears in at least one item; union over items)
    Returns {path: is_empty} where is_empty = value is None, {}, [] or "".
    """
    out = {}
    for k, v in obj.items():
        p = f"{prefix}{k}"
        out[p] = v is None or v == {} or v == [] or v == ""
        if isinstance(v, dict):
            out.update(all_paths(v, p + "."))
        elif isinstance(v, list) and v:
            out[f"{p}[]"] = False
            for item in v:
                if isinstance(item, dict):
                    for sub, empty in all_paths(item, f"{p}[].").items():
                        out[sub] = out.get(sub, True) and empty  # empty only if empty in every item
    return out


present, empty = Counter(), Counter()
for r in raw:
    for path, is_empty in all_paths(r).items():
        present[path] += 1
        empty[path] += is_empty

cov = pd.DataFrame({"present": pd.Series(present), "empty": pd.Series(empty)}).sort_index()
cov["missing"] = len(raw) - cov.present
cov["present_pct"] = (100 * cov.present / len(raw)).round(1)
cov["depth"] = cov.index.str.count(r"\.") 
pd.set_option("display.max_rows", 200)

In [22]:
cov

,present,empty,missing,present_pct,depth
agent_version,750,0,0,100.0,0
battery,650,0,100,86.7,0
battery.battery_present,650,0,100,86.7,1
battery.charging_status,650,0,100,86.7,1
battery.condition,650,0,100,86.7,1
battery.cycle_count,650,0,100,86.7,1
battery.full_charge_capacity,650,0,100,86.7,1
battery.percentage,650,0,100,86.7,1
collected_at,750,0,0,100.0,0
company_id,750,0,0,100.0,0
